In [ ]:
import os
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.openai import OpenAIEmbedding
import chromadb
from dotenv import load_dotenv

# Load environment variables from .env file (need parentheses to actually call the function)
load_dotenv()
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

2025-11-29 14:16:21,904 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-29 14:16:23,308 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-29 14:16:23,912 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


2025-11-29 14:16:35,571 - INFO - Backing off send_request(...) for 0.6s (requests.exceptions.ReadTimeout: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Read timed out. (read timeout=15))


In [ ]:
# Load all documents from the "documents" directory
# This reads PDFs, text files, etc. and converts them into Document objects
documents = SimpleDirectoryReader(input_dir="documents").load_data()

In [ ]:
# Create a persistent Chroma client that saves data to disk at "./chroma_db"
# This ensures the database persists between script runs
chroma_client = chromadb.PersistentClient(path="./chroma_db")

In [ ]:
# Get existing collection or create new one named "pdf_collection"
# Collections in Chroma are like tables in a database
chroma_collection = chroma_client.get_or_create_collection("pdf_collection")

In [ ]:
# Wrap the Chroma collection in LlamaIndex's ChromaVectorStore adapter
# This allows LlamaIndex to interact with Chroma's storage
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [ ]:
# Create a StorageContext that tells LlamaIndex where to store vectors
# This is the critical piece that connects the index to persistent storage
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
# Initialize OpenAI embedding model to convert text into vectors
# text-embedding-3-small is cost-effective and works well for most use cases
embed_model = OpenAIEmbedding(model="text-embedding-3-small")

In [ ]:
# Create the index from documents and store vectors in Chroma
# The storage_context ensures embeddings are saved to the persistent database
# Without storage_context, embeddings would only exist in memory
index = VectorStoreIndex.from_documents(
    documents, 
    storage_context=storage_context,
    embed_model=embed_model
)